# CS781 Phase 6 — Abstractive legal summarization from the Phase 1 Legal LM, model 1 (PyTorch, Kaggle GPU)
Phase 1 JAX weights → PyTorch port with KV-cache beam search → prefix-LM fine-tuning (document → separator → summary) with compressed long-document views → LoRMA / Adapters / AdaLoRA / full fine-tuning → distillation into a half-depth student → one `submission` zip per experiment.

## 1. Setup and configuration

In [1]:
import os, sys, time, json, math, random, pickle, zipfile, gc, copy, glob, subprocess, importlib.util
from collections import Counter
T0 = time.time()
if importlib.util.find_spec("regex") is None: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "regex"], check=False)
import numpy as np, regex as re2, torch, torch.nn as nn, torch.nn.functional as F
import multiprocessing as mp
from huggingface_hub import hf_hub_download, HfApi, login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF"); login(token=HF_TOKEN)
except Exception as e:
    HF_TOKEN = None; print("HF login skipped:", e)

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
HF_REPO = "Exploration-Lab/CS-781-Capstone"; WORK = "/kaggle/working/cs781_phase6_m1"; os.makedirs(WORK, exist_ok=True)
PIN = None
FOUND = [(p, pickle.load(open(p, "rb"))) for p in glob.glob("/kaggle/input/**/final_params.pkl", recursive=True)]
print("Phase 1 pickles found:", [(p, x["variant"]) for p, x in FOUND])
P1, d = next(((p, x) for p, x in FOUND if PIN in (None, p) and not any(k in x["cfg"] for k in ("kinds", "encdec", "adaln"))), (None, None)); del FOUND
assert d is not None, "no decoder-only ours_* final_params.pkl attached: add that Phase 1 run's output as a Kaggle dataset (or set PIN to its path)"
print("using", P1, "->", d["variant"])
TOKP = os.path.join(os.path.dirname(P1), "bpe_legal.json")
if not os.path.exists(TOKP): TOKP = glob.glob("/kaggle/input/**/bpe_legal.json", recursive=True)[0]
TOTAL_SEC, POST_SEC = 5.4 * 3600, 18 * 60
CTX, BS, NVAL = 1280, 8, 300
BEAMS, NR, ALPHA, MINL, GBS = 4, 3, 1.0, 30, 16
NG = torch.cuda.device_count()
EXPS = [("lorma", 1.0), ("adapter", 0.9), ("adalora", 1.0), ("full", 1.1), ("kd", 1.2)]
LR = dict(lorma=5e-4, adapter=5e-4, adalora=5e-4, full=8e-5, kd=1e-4)
print("gpus:", NG)

Phase 1 pickles found: [('/kaggle/input/datasets/nirbhaykachhatiya/phase2m1/final_params.pkl', 'ours_hybrid_gate')]
using /kaggle/input/datasets/nirbhaykachhatiya/phase2m1/final_params.pkl -> ours_hybrid_gate
gpus: 2


## 2. Data: Phase 6 files, hold-out split, Phase 1 byte-level BPE, three long-document views per case

In [2]:
PAT = re2.compile(r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+")
def b2u():
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256)); cs = bs[:]; n = 0
    for b in range(256):
        if b not in bs: bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))
B2U = b2u(); U2B = {v: k for k, v in B2U.items()}

class BPE:
    def __init__(s, path):
        d = json.load(open(path, encoding="utf-8")); s.itos = d["itos"]; s.rank = {tuple(m): i for i, m in enumerate(d["merges"])}; s.stoi = {t: i for i, t in enumerate(s.itos)}; s.c = {}
    def word(s, w):
        r = s.c.get(w)
        if r is None:
            x = [B2U[b] for b in w.encode("utf-8")]
            while len(x) > 1:
                best = min(zip(x, x[1:]), key=lambda q: s.rank.get(q, 1 << 60))
                if best not in s.rank: break
                a, b = best; y, i = [], 0
                while i < len(x):
                    if i < len(x) - 1 and x[i] == a and x[i + 1] == b: y.append(a + b); i += 2
                    else: y.append(x[i]); i += 1
                x = y
            r = s.c[w] = [s.stoi[t] for t in x]
        return r
    def encode(s, t): return [i for w in PAT.findall(t) for i in s.word(w)]
    def decode(s, ids): return bytearray(U2B[c] for t in ids if t > 3 for c in s.itos[t]).decode("utf-8", "replace")

TK = BPE(TOKP); EOS = 3; SEP = np.asarray([EOS] + TK.encode(" Summary:"), np.int64); print("EOS token string:", TK.itos[EOS], "| separator ids:", SEP.tolist())
def norm(t): return " ".join(t.split())
def encs(ds): return [np.asarray(TK.encode(norm(d["summary"])), np.uint16) for d in ds]

def views(text):
    ss = [s for s in re2.split(r"(?<=[.;?!])\s+", norm(text)) if s] or [""]
    ids = [np.asarray(TK.encode(("" if i == 0 else " ") + s), np.int64) for i, s in enumerate(ss)]; a = np.concatenate(ids).astype(np.uint16); n = len(a)
    if n <= DL: return a, a, a, n
    h = DL * 3 // 5; ht = np.concatenate([a[:h], a[n - (DL - h):]]); sg = DL // 8; ev = np.concatenate([a[s0:s0 + sg] for s0 in np.linspace(0, n - sg, 8).astype(int)])
    ws = [set(re2.findall(r"\p{L}{4,}", s.lower())) for s in ss]; df = Counter(w for x in ws for w in x); sc = [sum(df[w] for w in x) / (len(s.split()) + 5) for x, s in zip(ws, ss)]
    keep, used = set(), 0
    for j in np.argsort(sc)[::-1]:
        if used + len(ids[j]) <= DL: keep.add(int(j)); used += len(ids[j])
    return ht, ev, (np.concatenate([ids[j] for j in sorted(keep)]).astype(np.uint16) if keep else a[:DL]), n
def encd(ds): return [views(d["document"]) for d in ds]
def par(fn, ds):
    with mp.get_context("fork").Pool(os.cpu_count()) as pool: return [a for p in pool.imap(fn, [ds[i:i + 50] for i in range(0, len(ds), 50)]) for a in p]

files = HfApi().list_repo_files(HF_REPO, repo_type="dataset", token=HF_TOKEN)
def load(key):
    fn = next(f for f in files if f.startswith("Phase_6_Summ/") and f.endswith(key))
    d = json.load(open(hf_hub_download(HF_REPO, fn, repo_type="dataset", token=HF_TOKEN, local_dir="/tmp/hf"), encoding="utf-8"))
    return [{"id": k, **v} for k, v in d.items()] if isinstance(d, dict) else d

TOKC = f"{WORK}/tokens.pkl"
if os.path.exists(TOKC): TRD, TRS, VAD, VAS, VREF, TED, TE_ID, SL, DL, DLEN = pickle.load(open(TOKC, "rb"))
else:
    tr, te = load("train.json"), load("test.json"); perm = np.random.RandomState(0).permutation(len(tr)); nv = min(NVAL, max(len(tr) // 10, 1)); vi, ti = perm[:nv], perm[nv:]
    t0 = time.time(); S = par(encs, tr); SL = int(min(max(np.percentile([len(x) for x in S], 90), 128), 512)); DL = CTX - SL - len(SEP) - 1
    D_ = par(encd, tr); TED = [x[:3] for x in par(encd, te)]; DLEN = [x[3] for x in D_]
    TRD, TRS, VAD, VAS = [D_[i][:3] for i in ti], [S[i] for i in ti], [D_[i][:3] for i in vi], [S[i] for i in vi]; VREF = [norm(tr[i]["summary"]) for i in vi]; TE_ID = [d["id"] for d in te]
    pickle.dump((TRD, TRS, VAD, VAS, VREF, TED, TE_ID, SL, DL, DLEN), open(TOKC, "wb")); del tr, te, D_, S; gc.collect(); print(f"tokenised in {time.time()-t0:.0f}s")
NTR = len(TRD); MI = dict(ht=0, ev=1, sel=2); DOCMODE = "ev"
print(f"train/val/test {NTR}/{len(VAD)}/{len(TED)} | median doc tokens {np.median(DLEN):.0f} (share > {DL}: {np.mean(np.array(DLEN) > DL):.2f}) | summary tokens median {np.median([len(x) for x in TRS]):.0f}, p90 {np.percentile([len(x) for x in TRS], 90):.0f} -> SL={SL}, DL={DL}")

def mk(Dv, Sv, idx, mode):
    rows = [(np.concatenate([Dv[i][MI[mode]], SEP, Sv[i][:SL - 1], [EOS]]).astype(np.int64), len(Dv[i][MI[mode]]) + len(SEP)) for i in idx]
    ids = np.zeros((len(rows), max(len(r) for r, _ in rows)), np.int64); m = np.zeros(ids.shape, bool)
    for b, (r, st) in enumerate(rows): ids[b, :len(r)] = r; m[b, st:len(r)] = True
    return torch.from_numpy(ids).cuda(), torch.from_numpy(m).cuda()

EOS token string: </s> | separator ids: [3, 6596, 2863, 62]


Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

tokenised in 43s
train/val/test 6730/300/100 | median doc tokens 3791 (share > 763: 0.99) | summary tokens median 777, p90 1846 -> SL=512, DL=763


## 3. Model: Phase 1 decoder in PyTorch (tied LM head, KV-cache decoding) and adaptation modules

In [3]:
P, C = d["params"], d["cfg"]; assert d["variant"].startswith("ours_"), d["variant"]
lp = P["layers"]; L, D, HHD = lp["wq"].shape; HD = lp["qn"].shape[1]; H = HHD // HD; KV = lp["wk"].shape[2] // HD; V = P["emb"].shape[0]; R = H // KV
WIN = 256; LOC = [bool(C["hybrid"]) and i % 3 != 2 for i in range(L)]; LINS = ("wq", "wk", "wv", "wo", "w1", "w3", "w2")
INV = torch.tensor(1.0 / (10000.0 ** (np.arange(0, HD, 2) / HD)), dtype=torch.float32)
print(d["variant"], f"L={L} D={D} H={H} KV={KV} HD={HD} V={V}", "|", torch.cuda.get_device_name(0))

def f32(w): return torch.from_numpy(np.asarray(w, np.float32).copy())
def lin(w):
    m = nn.Linear(w.shape[0], w.shape[1], bias=False); m.weight.data = torch.from_numpy(np.ascontiguousarray(w.T, dtype=np.float32)); return m

class RMS(nn.Module):
    def __init__(s, w): super().__init__(); s.w = nn.Parameter(f32(w))
    def forward(s, x): x = x.float(); return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * s.w

def rope(x, cos, sin):
    a, b = x[..., :HD // 2].float(), x[..., HD // 2:].float()
    return torch.cat([a * cos - b * sin, b * cos + a * sin], -1)

class Blk(nn.Module):
    def __init__(s, i):
        super().__init__(); s.loc = LOC[i]
        for n in LINS: setattr(s, n, lin(lp[n][i]))
        s.n1, s.n2, s.qn, s.kn = RMS(lp["n1"][i]), RMS(lp["n2"][i]), RMS(lp["qn"][i]), RMS(lp["kn"][i]); s.g = nn.Parameter(f32(lp["g"][i])) if "g" in lp else None
    def gate(s, j): return 1.0 if s.g is None else s.g[j]
    def qkv(s, h, cos, sin):
        B, T, _ = h.shape; q, k, v = s.wq(h).view(B, T, H, HD), s.wk(h).view(B, T, KV, HD), s.wv(h).view(B, T, KV, HD)
        return rope(s.qn(q), cos, sin).to(v.dtype), rope(s.kn(k), cos, sin).to(v.dtype), v
    def out(s, x, o):
        x = x + s.gate(0) * s.wo(o); h = s.n2(x); return x + s.gate(1) * s.w2(F.silu(s.w1(h)) * s.w3(h))
    def attn(s, q, k, v, mask, causal): return F.scaled_dot_product_attention(q.transpose(1, 2), k.repeat_interleave(R, 2).transpose(1, 2), v.repeat_interleave(R, 2).transpose(1, 2), attn_mask=mask, is_causal=causal)
    def forward(s, x, cos, sin, lm):
        q, k, v = s.qkv(s.n1(x), cos, sin); B, T = x.shape[:2]
        return s.out(x, s.attn(q, k, v, lm if s.loc else None, not s.loc).transpose(1, 2).reshape(B, T, H * HD))
    def prefill(s, x, cos, sin, am):
        q, k, v = s.qkv(s.n1(x), cos, sin); B, T = x.shape[:2]
        return s.out(x, s.attn(q, k, v, am, False).transpose(1, 2).reshape(B, T, H * HD)), k.transpose(1, 2), v.transpose(1, 2)
    def step(s, x, cos, sin, kc, vc, t, valid):
        n = x.shape[0]; q, k, v = s.qkv(s.n1(x), cos, sin); kc[:, :, t] = k[:, 0]; vc[:, :, t] = v[:, 0]; lo = max(0, t + 1 - WIN) if s.loc else 0
        a = (q[:, 0].view(n, KV, R, HD) @ kc[:, :, lo:t + 1].transpose(-1, -2)) * HD ** -0.5
        a = a.masked_fill(~valid[:, None, None, lo:t + 1], float("-inf")).float().softmax(-1).to(v.dtype)
        return s.out(x, (a @ vc[:, :, lo:t + 1]).reshape(n, 1, H * HD))

class LM(nn.Module):
    def __init__(s, keep):
        super().__init__(); s.emb = nn.Embedding(V, D); s.emb.weight.data = f32(P["emb"]); s.blocks = nn.ModuleList([Blk(i) for i in keep]); s.nf = RMS(P["nf"])
        t = torch.arange(CTX)[:, None] * INV[None]; s.register_buffer("cos", t.cos(), persistent=False); s.register_buffer("sin", t.sin(), persistent=False)
    def logits(s, ids, mt):
        T = ids.shape[1]; x = s.emb(ids); cos, sin = s.cos[:T][None, :, None], s.sin[:T][None, :, None]
        lm = torch.ones(T, T, dtype=torch.bool, device=ids.device).tril().triu(1 - WIN)
        for b in s.blocks: x = b(x, cos, sin, lm)
        return s.nf(x)[:, :-1][mt] @ s.emb.weight.t()
    def prefill(s, ids, pad):
        B, T = ids.shape; dev = ids.device; ar = torch.arange(T, device=dev)[None]; pos = (ar - pad[:, None]).clamp(min=0); cos, sin = s.cos[pos][:, :, None], s.sin[pos][:, :, None]
        gm = ((torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None] & (ar >= pad[:, None])[:, None, :]) | torch.eye(T, dtype=torch.bool, device=dev)[None])[:, None]
        lm = gm & torch.ones(T, T, dtype=torch.bool, device=dev).triu(1 - WIN)[None, None]; x = s.emb(ids); kv = []
        for b in s.blocks: x, k, v = b.prefill(x, cos, sin, lm if b.loc else gm); kv.append((k, v))
        return s.nf(x[:, -1]) @ s.emb.weight.t(), kv
    def step(s, tok, pos, kc, vc, t, valid):
        x = s.emb(tok); cos, sin = s.cos[pos][:, None, None], s.sin[pos][:, None, None]
        for i, b in enumerate(s.blocks): x = b.step(x, cos, sin, kc[i], vc[i], t, valid)
        return s.nf(x)[:, 0] @ s.emb.weight.t()

class Wrap(nn.Module):
    def __init__(s, lm, teacher=None): super().__init__(); s.lm = lm; s.teacher = teacher
    def forward(s, ids, m):
        mt = m[:, 1:]; z = s.lm.logits(ids, mt).float(); ce = F.cross_entropy(z, ids[:, 1:][mt], reduction="sum"); kl = torch.zeros_like(ce)
        if s.teacher is not None:
            with torch.no_grad(): tz = s.teacher.logits(ids, mt).float()
            kl = F.kl_div(F.log_softmax(z / 2, -1), F.softmax(tz / 2, -1), reduction="sum")
        return torch.stack([ce, kl, mt.sum().float()])

class LoRMA(nn.Module):
    def __init__(s, b, r=16): super().__init__(); s.b = b; o = b.out_features; s.A = nn.Parameter(torch.randn(o, r) * o ** -0.5); s.B = nn.Parameter(torch.zeros(r, o))
    def forward(s, x): y = s.b(x); return y + (y @ s.A) @ s.B

class Adapter(nn.Module):
    def __init__(s, b, r=64):
        super().__init__(); s.b = b; o = b.out_features; s.d, s.u = nn.Linear(o, r), nn.Linear(r, o); nn.init.zeros_(s.u.weight); nn.init.zeros_(s.u.bias)
    def forward(s, x): y = s.b(x); return y + s.u(F.gelu(s.d(y)))

class AdaLoRA(nn.Module):
    def __init__(s, b, r=12):
        super().__init__(); s.b = b; i, o = b.in_features, b.out_features
        s.P, s.Q, s.E = nn.Parameter(torch.empty(o, r)), nn.Parameter(torch.empty(r, i)), nn.Parameter(torch.zeros(r))
        nn.init.orthogonal_(s.P); nn.init.orthogonal_(s.Q); s.register_buffer("mk", torch.ones(r)); s.register_buffer("imp", torch.zeros(r))
    def forward(s, x): return s.b(x) + ((x @ s.Q.t()) * (s.E * s.mk)) @ s.P.t()

def orth(a):
    I = torch.eye(a.E.numel(), device=a.E.device); return ((a.P.t() @ a.P - I) ** 2).sum() + ((a.Q @ a.Q.t() - I) ** 2).sum()

def ada_step(ada, step, total, b0):
    for a in ada: a.imp.mul_(0.85).add_(0.15 * (a.E * a.E.grad).abs().detach())
    ti, tf = int(0.1 * total), int(0.7 * total)
    if ti <= step <= tf and (step % 25 == 0 or step == tf):
        bud = int(b0 // 2 + (b0 - b0 // 2) * (1 - (step - ti) / max(tf - ti, 1)) ** 3)
        thr = torch.cat([a.imp for a in ada]).topk(bud).values[-1]
        for a in ada: a.mk.copy_((a.imp >= thr).float())

def adapt(m, kind):
    if kind in ("full", "kd"): return m
    for p in m.parameters(): p.requires_grad = False
    for b in m.blocks:
        for n in LINS:
            l = getattr(b, n)
            if kind == "lorma": setattr(b, n, LoRMA(l))
            elif kind == "adalora": setattr(b, n, AdaLoRA(l))
        if kind == "adapter":
            for n in ("wo", "w2"): setattr(b, n, Adapter(getattr(b, n)))
    return m

def build(kind): return adapt(LM(range(0, L, 2) if kind == "kd" else range(L)), kind)

ours_hybrid_gate L=12 D=768 H=12 KV=4 HD=64 V=16384 | Tesla T4


## 4. Generation (batched KV-cache beam search with n-gram blocking), ROUGE, and the training loop

In [4]:
@torch.no_grad()
def beam(m, ids, pad, nb, maxlen, nr):
    B, T = ids.shape; dev = ids.device; n = B * nb; Tm = T + maxlen
    with torch.autocast("cuda", torch.float16):
        lg, kv = m.prefill(ids, pad)
        lp = F.log_softmax(lg.float(), -1); lp[:, EOS] = -1e9; sc, tok = lp.topk(nb, -1)
        kc, vc = [k.new_zeros(n, KV, Tm, HD) for k, _ in kv], [v.new_zeros(n, KV, Tm, HD) for _, v in kv]
        for (k, v), a, b in zip(kv, kc, vc): a[:, :, :T] = k.repeat_interleave(nb, 0); b[:, :, :T] = v.repeat_interleave(nb, 0)
        valid = torch.ones(n, Tm, dtype=torch.bool, device=dev); valid[:, :T] = (torch.arange(T, device=dev)[None] >= pad[:, None]).repeat_interleave(nb, 0)
        seq, sc, fin, base = tok.view(n, 1), sc.view(n), tok.view(n) == EOS, (T - pad).repeat_interleave(nb)
        for t in range(maxlen - 1):
            if fin.all(): break
            lp = F.log_softmax(m.step(seq[:, -1:], base + t, kc, vc, T + t, valid).float(), -1)
            if t + 2 < MINL: lp[:, EOS] = -1e9
            Ls = seq.shape[1]
            if nr and Ls >= nr:
                c = nr - 1; mt = torch.ones(n, Ls - c, dtype=torch.bool, device=dev)
                for j in range(c): mt &= seq[:, j:Ls - c + j] == seq[:, Ls - c + j:Ls - c + j + 1]
                rr, ci = mt.nonzero(as_tuple=True); lp[rr, seq[:, c:][rr, ci]] = -1e9
            lp[fin] = -1e9; lp[fin, EOS] = 0.0
            sc2, ix = (sc[:, None] + lp).view(B, nb * V).topk(nb, -1); rows = (torch.arange(B, device=dev)[:, None] * nb + ix // V).view(-1)
            seq = torch.cat([seq[rows], (ix % V).view(n, 1)], 1); sc = sc2.view(n); fin = fin[rows] | (seq[:, -1] == EOS)
            kc, vc = [x.index_select(0, rows) for x in kc], [x.index_select(0, rows) for x in vc]
    Ln = ((seq == EOS).cumsum(1) == 0).sum(1) + 1; best = (sc / Ln.float() ** ALPHA).view(B, nb).argmax(1); out = []
    for b in range(B):
        s_ = seq[b * nb + best[b]].tolist(); out.append(s_[:s_.index(EOS)] if EOS in s_ else s_)
    return out

def generate(m, docs, beams=BEAMS, maxlen=None, nr=NR, bs=GBS):
    m.cuda().eval(); maxlen = maxlen or SL; res = [None] * len(docs); order = np.argsort([len(x) for x in docs])
    for i in range(0, len(docs), bs):
        sel = order[i:i + bs]; pr = [np.concatenate([docs[j], SEP]).astype(np.int64) for j in sel]; T = max(len(x) for x in pr); ids = np.zeros((len(pr), T), np.int64)
        for b, x in enumerate(pr): ids[b, T - len(x):] = x
        pad = torch.tensor([T - len(x) for x in pr]).cuda()
        for j, o in zip(sel, beam(m, torch.from_numpy(ids).cuda(), pad, beams, maxlen, nr)): res[j] = TK.decode(o)
    return res

def toks(t): return re2.findall(r"\p{L}+|\p{N}+", t.lower())
def ngr(x, n): return Counter(tuple(x[i:i + n]) for i in range(len(x) - n + 1))
def lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b): cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]
def rouge(c, r):
    a, b = toks(c), toks(r); f = lambda o, x, y: 2 * o / (x + y) if o else 0.0
    return [f(sum((ngr(a, n) & ngr(b, n)).values()), max(len(a) - n + 1, 0), max(len(b) - n + 1, 0)) for n in (1, 2)] + [f(lcs(a, b), len(a), len(b))]
def score(hy, refs):
    r = np.mean([rouge(h, x) for h, x in zip(hy, refs)], 0); rep = np.mean([1 - len(g) / max(sum(g.values()), 1) for g in (ngr(toks(h), 3) for h in hy)])
    return dict(r1=float(r[0]), r2=float(r[1]), rl=float(r[2]), gen_words=float(np.mean([len(toks(h)) for h in hy])), ref_words=float(np.mean([len(toks(x)) for x in refs])), rep3=float(rep))
def evaluate(m, Dv, refs, mode, **kw):
    t0 = time.time(); hy = generate(m, [x[MI[mode]] for x in Dv], **kw); return dict(score(hy, refs), gen_s=time.time() - t0), hy

@torch.no_grad()
def vloss(m, n=96):
    w = Wrap(m).cuda().eval(); tot = np.zeros(3)
    for i in range(0, min(n, len(VAD)), BS): ids, mm = mk(VAD, VAS, range(i, min(i + BS, n, len(VAD))), DOCMODE); tot += w(ids, mm).cpu().numpy()
    return float(tot[0] / tot[2])

def fit(name, m, kind, budget, teacher=None):
    m.cuda().train(); w = Wrap(m, teacher); dm = nn.DataParallel(w) if NG > 1 else w
    tp = [p for p in m.parameters() if p.requires_grad]; opt = torch.optim.AdamW(tp, lr=LR[kind], weight_decay=0.01, betas=(0.9, 0.98)); sc = torch.amp.GradScaler("cuda")
    ada = [x for x in m.modules() if isinstance(x, AdaLoRA)]; b0 = sum(a.E.numel() for a in ada)
    step, total, tb, ptr, perm, ema, hist, t0 = 0, 10 ** 9, None, 0, np.random.permutation(NTR), None, [], time.time()
    torch.cuda.reset_peak_memory_stats()
    while step < total and time.time() - T0 < TOTAL_SEC - POST_SEC:
        if ptr + BS > len(perm): perm, ptr = np.random.permutation(NTR), 0
        idx = perm[ptr:ptr + BS]; ptr += BS; ids, mm = mk(TRD, TRS, idx, DOCMODE)
        f = min(1.0, (step + 1) / 100) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(step / total, 1.0))))
        for g in opt.param_groups: g["lr"] = LR[kind] * f
        with torch.autocast("cuda", torch.float16):
            o = dm(ids, mm).view(-1, 3).sum(0); loss = o[0] / o[2]
            if teacher is not None: loss = 0.5 * loss + 2.0 * o[1] / o[2]
        if ada: loss = loss + 0.1 * sum(orth(a) for a in ada)
        opt.zero_grad(set_to_none=True); sc.scale(loss).backward(); sc.unscale_(opt); gn = nn.utils.clip_grad_norm_(tp, 1.0)
        if ada and torch.isfinite(gn): ada_step(ada, step, total, b0)
        sc.step(opt); sc.update(); step += 1
        if total == 10 ** 9:
            if step == 5: torch.cuda.synchronize(); tb = time.time()
            if step == 25:
                torch.cuda.synchronize(); rate = (time.time() - tb) / 20; total = step + int(0.97 * (budget - (time.time() - t0)) / rate)
                print(f"[{name}] {rate:.3f}s/step -> total_steps={total} (~{total * BS / NTR:.2f} epochs)", flush=True)
        if step % 10 == 0: ema = loss.item() if ema is None else 0.9 * ema + 0.1 * loss.item()
        if step % 50 == 0 and ema is not None: hist.append((step, ema))
        if step % 200 == 0: print(f"[{name}] step {step}/{total} loss {ema:.4f} | {(time.time() - t0) / 60:.1f} min", flush=True)
        if total < 10 ** 9 and step < total and step % max(total // 5, 1) == 0: m.eval(); print(f"[{name}] step {step} val CE {vloss(m):.4f}", flush=True); m.train()
    tt = time.time() - t0; mem = torch.cuda.max_memory_allocated() / 2 ** 30; del opt, sc, dm, w; gc.collect(); torch.cuda.empty_cache(); m.eval()
    vl = vloss(m); ev, hy = evaluate(m, VAD, VREF, DOCMODE); t1 = time.time(); ts = generate(m, [x[MI[DOCMODE]] for x in TED]); tg = time.time() - t1
    os.makedirs(f"{WORK}/{name}", exist_ok=True); pj = f"{WORK}/{name}/prediction.json"
    json.dump([{"id": i, "summary": s} for i, s in zip(TE_ID, ts)], open(pj, "w"))
    with zipfile.ZipFile(f"{WORK}/sub_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
    json.dump([{"ref": r, "hyp": h} for r, h in zip(VREF[:50], hy[:50])], open(f"{WORK}/{name}/val_samples.json", "w"), indent=1)
    torch.save({n: p.detach().cpu() for n, p in m.named_parameters() if p.requires_grad}, f"{WORK}/{name}/weights.pt")
    res = dict(name=name, **ev, val_ce=vl, sel=(ev["r1"] + ev["rl"]) / 2, params_M=sum(p.numel() for p in m.parameters()) / 1e6, train_M=sum(p.numel() for p in tp) / 1e6, steps=step, train_min=tt / 60, mem_GB=mem, test_gen_s=tg, hist=hist)
    print(f"[{name}] holdout R1 {ev['r1']:.4f} R2 {ev['r2']:.4f} RL {ev['rl']:.4f} | words gen/ref {ev['gen_words']:.0f}/{ev['ref_words']:.0f} | rep3 {ev['rep3']:.3f} | val CE {vl:.3f} | gen {ev['gen_s']:.0f}s/{len(VAD)} docs", flush=True); return res, m

## 5. Experiments (time-shared budget; KD teacher = best holdout (R1+RL)/2 model so far, student = every-other-layer Phase 1 init)

In [5]:
RES, BEST = {}, (-1.0, None)
for i, (name, w) in enumerate(EXPS):
    alloc = (TOTAL_SEC - (time.time() - T0)) * w / sum(x[1] for x in EXPS[i:])
    if alloc < 25 * 60: print("skip", name, "(not enough time)"); continue
    print(f"=== {name}: {alloc / 60:.0f} min allocated, {(time.time() - T0) / 60:.0f} min elapsed", flush=True)
    assert name != "kd" or BEST[1] is not None
    tc = BEST[1].cuda().eval() if name == "kd" else None
    if tc is not None:
        for p in tc.parameters(): p.requires_grad = False
    RES[name], m = fit(name, build(name), name, alloc - POST_SEC, tc)
    if name != "kd" and RES[name]["sel"] > BEST[0]: BEST = (RES[name]["sel"], copy.deepcopy(m).cpu())
    if tc is not None: tc.cpu()
    m.cpu(); del m, tc; gc.collect(); torch.cuda.empty_cache(); json.dump(RES, open(f"{WORK}/results.json", "w"), indent=1)

print(f"{'exp':9s}{'total M':>9s}{'train M':>9s}{'R1':>8s}{'R2':>8s}{'RL':>8s}{'words':>7s}{'rep3':>7s}{'valCE':>7s}{'min':>7s}{'GB':>6s}{'gen s':>7s}")
for k, r in RES.items(): print(f"{k:9s}{r['params_M']:9.1f}{r['train_M']:9.2f}{r['r1']:8.4f}{r['r2']:8.4f}{r['rl']:8.4f}{r['gen_words']:7.0f}{r['rep3']:7.3f}{r['val_ce']:7.3f}{r['train_min']:7.1f}{r['mem_GB']:6.1f}{r['gen_s']:7.0f}")
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.figure(figsize=(9, 5))
for k, r in RES.items():
    if r["hist"]: plt.plot(*zip(*r["hist"]), label=k)
plt.xlabel("step"); plt.ylabel("train loss (EMA)"); plt.grid(True); plt.legend(); plt.savefig(f"{WORK}/loss_curves.png", dpi=200, bbox_inches="tight"); plt.show()
print("submissions:", sorted(glob.glob(f"{WORK}/sub_*.zip")))

=== lorma: 62 min allocated, 1 min elapsed
[lorma] 0.427s/step -> total_steps=6009 (~7.14 epochs)
[lorma] step 200/6009 loss 1.9201 | 1.5 min
[lorma] step 400/6009 loss 1.8144 | 2.9 min
[lorma] step 600/6009 loss 1.8654 | 4.4 min
[lorma] step 800/6009 loss 1.8063 | 5.9 min
[lorma] step 1000/6009 loss 1.7082 | 7.4 min
[lorma] step 1200/6009 loss 1.7247 | 8.9 min
[lorma] step 1201 val CE 1.8335
[lorma] step 1400/6009 loss 1.7534 | 10.6 min
[lorma] step 1600/6009 loss 1.7947 | 12.1 min
[lorma] step 1800/6009 loss 1.6541 | 13.5 min
[lorma] step 2000/6009 loss 1.6483 | 15.0 min
[lorma] step 2200/6009 loss 1.6748 | 16.5 min
[lorma] step 2400/6009 loss 1.6906 | 18.0 min
[lorma] step 2402 val CE 1.8536
[lorma] step 2600/6009 loss 1.6026 | 19.7 min
[lorma] step 2800/6009 loss 1.5831 | 21.2 min
[lorma] step 3000/6009 loss 1.6105 | 22.6 min
[lorma] step 3200/6009 loss 1.6253 | 24.1 min
[lorma] step 3400/6009 loss 1.5654 | 25.6 min
[lorma] step 3600/6009 loss 1.5259 | 27.1 min
[lorma] step 3603 va

## 6. Analysis on the best non-KD model: long-document view, decoding, and generation length (150 hold-out documents)

In [6]:
bm = BEST[1].cuda().eval(); sub = list(range(min(150, len(VAD)))); Dv, Rv = [VAD[i] for i in sub], [VREF[i] for i in sub]
runs = [(f"view={v}", v, {}) for v in ("ev", "ht", "sel")] + [("greedy", DOCMODE, dict(beams=1)), ("no n-gram block", DOCMODE, dict(nr=0)), (f"max len {SL // 2}", DOCMODE, dict(maxlen=SL // 2)), (f"max len {SL}", DOCMODE, dict(maxlen=SL))]
print(f"{'setting':18s}{'R1':>8s}{'R2':>8s}{'RL':>8s}{'gen words':>11s}{'ref words':>11s}{'rep3':>7s}{'s':>6s}")
for nm, v, kw in runs:
    r, _ = evaluate(bm, Dv, Rv, v, **kw); print(f"{nm:18s}{r['r1']:8.4f}{r['r2']:8.4f}{r['rl']:8.4f}{r['gen_words']:11.0f}{r['ref_words']:11.0f}{r['rep3']:7.3f}{r['gen_s']:6.0f}")
bm.cpu()

setting                 R1      R2      RL  gen words  ref words   rep3     s
view=ev             0.4333  0.1995  0.2199        368        914  0.015   218
view=ht             0.4238  0.2053  0.2228        354        914  0.014   217
view=sel            0.4492  0.2127  0.2233        378        914  0.017   218
greedy              0.4210  0.1856  0.2181        367        914  0.038   187
no n-gram block     0.3332  0.1255  0.2099        405        914  0.677   218
max len 256         0.3362  0.1598  0.1913        216        914  0.014   107
max len 512         0.4333  0.1995  0.2199        368        914  0.015   218


LM(
  (emb): Embedding(16384, 768)
  (blocks): ModuleList(
    (0-11): 12 x Blk(
      (wq): AdaLoRA(
        (b): Linear(in_features=768, out_features=768, bias=False)
      )
      (wk): AdaLoRA(
        (b): Linear(in_features=768, out_features=256, bias=False)
      )
      (wv): AdaLoRA(
        (b): Linear(in_features=768, out_features=256, bias=False)
      )
      (wo): AdaLoRA(
        (b): Linear(in_features=768, out_features=768, bias=False)
      )
      (w1): AdaLoRA(
        (b): Linear(in_features=768, out_features=2048, bias=False)
      )
      (w3): AdaLoRA(
        (b): Linear(in_features=768, out_features=2048, bias=False)
      )
      (w2): AdaLoRA(
        (b): Linear(in_features=2048, out_features=768, bias=False)
      )
      (n1): RMS()
      (n2): RMS()
      (qn): RMS()
      (kn): RMS()
    )
  )
  (nf): RMS()
)